# Séance 2 — Penser les données avant Pandas : grain, faits, dimensions et qualité

**Decision problem:** comment éviter qu’un code parfaitement valide produise un chiffre d’affaires faux ?

Cette séance utilise Python comme **outil de vérification**. Le vrai sujet est la structure des données.

## Objectifs
À la fin de la séance, vous devez savoir :
- définir le **grain** d’une table ;
- distinguer **fait**, **dimension**, mesure et attribut ;
- reconnaître les cardinalités `1:1`, `1:N`, `N:N` ;
- construire un petit schéma en étoile ;
- contrôler une jointure au lieu de lui faire confiance ;
- expliquer pourquoi « le code tourne » ne signifie pas « le résultat est juste ».


## 1 — Avant Python : que représente UNE ligne ?

Une équipe e-commerce vous donne quatre fichiers. Avant de les ouvrir, formulez le grain attendu :

| Table | Grain attendu | Rôle probable |
|---|---|---|
| customers | ? | ? |
| products | ? | ? |
| orders | ? | ? |
| order_lines | ? | ? |

**Règle :** si vous ne pouvez pas finir la phrase « une ligne représente… », vous n’êtes pas prêt à joindre la table.


In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "datasets").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT / "datasets" / "retail_case"

customers = pd.read_csv(DATA / "customers.csv")
products = pd.read_csv(DATA / "products.csv")
orders = pd.read_csv(DATA / "orders.csv", parse_dates=["order_date"])
order_lines = pd.read_csv(DATA / "order_lines.csv")

for name, df in {"customers": customers, "products": products, "orders": orders, "order_lines": order_lines}.items():
    print(f"{name:12} rows={len(df):2}  columns={list(df.columns)}")


## 2 — Faits et dimensions

Une **dimension** décrit le contexte : *qui ? quoi ? où ? quand ?*  
Une **table de faits** enregistre des événements mesurables à un grain précis.

Pour notre analyse de ventes :

```text
DIM_CUSTOMER ─┐
              │
DIM_PRODUCT ──┼── FACT_SALES ── DIM_DATE
              │
DIM_CHANNEL ──┘
```

Ici, le grain naturel de `FACT_SALES` est **une ligne de commande**. `quantity`, `unit_price` et `revenue` sont des mesures.

> Question : pourquoi « une commande » serait-elle un mauvais grain si une commande peut contenir plusieurs produits ?


In [ ]:
# Construisons la table de faits en déclarant nos hypothèses de cardinalité.
fact_sales = (
    order_lines
    .merge(orders, on="order_id", how="left", validate="many_to_one")
    .merge(customers, on="customer_id", how="left", validate="many_to_one")
    .merge(products[["product_id", "product_name", "category"]], on="product_id", how="left", validate="many_to_one")
)
fact_sales["revenue"] = fact_sales["quantity"] * fact_sales["unit_price"]

print("Grain: one row per order line")
print("Rows:", len(fact_sales))
print("Revenue:", fact_sales["revenue"].sum())
fact_sales.head()


## 3 — Le piège : une jointure peut réussir et être fausse

On vous fournit maintenant une table de tags marketing où un client peut avoir plusieurs tags. Si on la joint directement à `FACT_SALES`, certaines ventes seront dupliquées. Pandas ne considère pas cela comme une erreur.


In [ ]:
customer_tags = pd.DataFrame({
    "customer_id": ["C001", "C001", "C002", "C002", "C003", "C003", "C004", "C004", "C005", "C005"],
    "tag": ["newsletter", "vip", "newsletter", "promo", "organic", "loyalty", "partner", "vip", "organic", "promo"]
})

correct_revenue = fact_sales["revenue"].sum()
wrong = fact_sales.merge(customer_tags, on="customer_id", how="left")
wrong_revenue = wrong["revenue"].sum()

print("Revenue before join:", correct_revenue)
print("Revenue after naive join:", wrong_revenue)
print("Rows before / after:", len(fact_sales), "/", len(wrong))
print("Inflation:", round((wrong_revenue / correct_revenue - 1) * 100, 1), "%")


### Pourquoi ?

La jointure a changé le **grain**. Une vente appartenant à un client avec deux tags apparaît deux fois.

Essayez maintenant de demander explicitement à Pandas la cardinalité que vous pensiez avoir :


In [ ]:
try:
    fact_sales.merge(customer_tags, on="customer_id", how="left", validate="many_to_one")
except Exception as e:
    print(type(e).__name__ + ":", e)


## 4 — Contrat de données minimal

Avant une analyse, écrivez vos invariants. Quelques contrôles valent souvent plus qu’un modèle sophistiqué.


In [ ]:
checks = {
    "order_id_not_null": orders["order_id"].notna().all(),
    "order_id_unique": orders["order_id"].is_unique,
    "customer_id_unique": customers["customer_id"].is_unique,
    "product_id_unique": products["product_id"].is_unique,
    "quantity_positive": (order_lines["quantity"] > 0).all(),
    "all_orders_matched": fact_sales["customer_id"].notna().all(),
}

pd.Series(checks, name="passed")


## 5 — Mini-challenge

Sans demander à une IA de décider à votre place :

1. Écrivez le grain de chacune des quatre tables en une phrase.
2. Identifiez les dimensions et le fait.
3. Expliquez pourquoi `customer_tags` crée un risque de duplication.
4. Proposez **deux** manières correctes d’utiliser les tags sans gonfler le CA.
5. Ajoutez un contrôle Python qui détecterait une anomalie avant publication d’un dashboard.

Vous pouvez utiliser une IA pour générer la syntaxe, **mais vous devez être capable d’expliquer le résultat et de vérifier le grain avant/après**.


## 6 — À retenir

- **Grain avant code.**
- Une table de faits enregistre des événements mesurables ; les dimensions donnent leur contexte.
- Une jointure est une hypothèse métier sur une cardinalité, pas seulement une commande Pandas.
- Vérifiez nombre de lignes, unicité des clés, valeurs manquantes et agrégats avant/après une jointure.
- Une IA peut écrire `merge`; la responsabilité du modèle de données et de la validation reste à l’analyste.

**Livrable projet :** un mini dictionnaire de données indiquant pour chaque table son grain, sa clé, son rôle et au moins deux règles de qualité.
